In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate torch evaluate scikit-learn


In [ ]:
import torch
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
import evaluate
from tqdm import tqdm
import os

In [ ]:
# Загрузка датасета
squad = load_dataset("squad_v2")
print(f"Train size: {len(squad['train'])}")
print(f"Validation size: {len(squad['validation'])}")

# Разбиение train на train/val (90/10)
train_data = squad["train"].shuffle(seed=42)
train_val_split = train_data.train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train": train_val_split["train"],
    "validation": train_val_split["test"]
})

print(f"New train size: {len(dataset['train'])}")
print(f"New validation size: {len(dataset['validation'])}")

In [ ]:
def format_prompt(context, question, answer=None):
    """Форматирование промпта для QA задачи"""
    prompt = f"""Context: {context}
Question: {question}
Answer: """
    
    if answer is not None:
        # Для обучения: добавляем ответ
        if isinstance(answer, dict):
            # SQuAD формат ответа
            answer_text = answer["text"][0] if answer["text"] else "No answer"
        else:
            answer_text = answer
        prompt += answer_text
    
    return prompt

def preprocess_function(examples):
    """Токенизация датасета"""
    prompts = []
    for context, question in zip(examples["context"], examples["question"]):
        prompts.append(format_prompt(context, question))
    
    # Токенизация промптов
    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )
    
    # Получение ответов для labels
    answers = []
    for ans in examples["answers"]:
        if ans["text"]:
            answers.append(ans["text"][0])
        else:
            answers.append("No answer")
    
    # Токенизация ответов
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            answers,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors=None
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Имя модели
model_name = "meta-llama/Llama-3.2-1B"

# Конфигурация 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Загрузка модели с 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA конфигурация
lora_config = LoraConfig(
    r=16,  # rank
    lora_alpha=32,  # scaling factor
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # целевые модули
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Должно показать ~0.5-1% trainable params

# Включение градиентного checkpointing для экономии памяти
model.config.use_cache = False
model.gradient_checkpointing_enable()

In [ ]:
# Применение токенизации к датасету
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Установка формата для PyTorch
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
# Data collator для динамического padding
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Аргументы обучения
training_args = TrainingArguments(
    output_dir="./squad-llama-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",  # Отключаем wandb/tensorboard для простоты
    push_to_hub=False,  # Пока False, загрузим позже
)

# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

In [ ]:
# Обучение
print("Starting training...")
trainer.train()

# Сохранение модели
model.save_pretrained("./lora-squad-model")
tokenizer.save_pretrained("./lora-squad-model")

print("Training completed!")